# GUI Element Detection — Classical Approach (OpenCV)

Detection using classical image processing (OpenCV), without deep learning.

Serves as a baseline to demonstrate the limitations of traditional approaches and justify the use of deep learning models.

**Pipeline:**
1. Load image in grayscale
2. Smooth with Gaussian Blur (reduce noise)
3. Detect edges with Canny Edge Detection
4. Close gaps with morphological operation (close)
5. Find contours and filter by minimum area
6. Draw bounding boxes on detected contours

In [ ]:
import os
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

## Configuration

In [ ]:
IMAGE_PATH   = "/content/drive/MyDrive/dataset_split/test/images/gui_example.png"
CANNY_MIN    = 70
CANNY_MAX    = 200
BLUR_KERNEL  = (5, 5)
MORPH_KERNEL = (3, 3)
MIN_AREA     = 100
OUTPUT_DIR   = "../results/opencv"

## Detection Pipeline

In [ ]:
def detect_gui_elements(image_path, canny_min=70, canny_max=200,
                         blur_kernel=(5, 5), morph_kernel=(3, 3),
                         min_area=100):
    """
    Detects potential GUI elements using classical image processing.

    Returns:
        image:    original image (grayscale)
        stages:   dictionary with intermediate pipeline images
        contours: list of filtered contours
        bboxes:   list of bounding boxes (x, y, w, h)
    """
    image = cv.imread(image_path, cv.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    blurred = cv.GaussianBlur(image, blur_kernel, 0)
    edges = cv.Canny(blurred, canny_min, canny_max)

    kernel = cv.getStructuringElement(cv.MORPH_RECT, morph_kernel)
    closed = cv.morphologyEx(edges, cv.MORPH_CLOSE, kernel)

    contours, _ = cv.findContours(closed, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    filtered = [cnt for cnt in contours if cv.contourArea(cnt) > min_area]
    bboxes = [cv.boundingRect(cnt) for cnt in filtered]

    stages = {
        "original": image, "blurred": blurred,
        "edges": edges, "closed": closed,
    }
    return image, stages, filtered, bboxes

## Visualisation

In [ ]:
def plot_stages(stages, output_path=None):
    """Shows the 4 intermediate pipeline stages side by side."""
    titles = [
        ("original", "1. Original (Grey)"), ("blurred", "2. Gaussian Blur"),
        ("edges", "3. Canny Edges"), ("closed", "4. Morphology (Close)"),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for i, (key, title) in enumerate(titles):
        axes[i].imshow(stages[key], cmap="gray")
        axes[i].set_title(title, fontsize=12)
        axes[i].axis("off")
    fig.suptitle("Image Processing Pipeline — OpenCV", fontsize=14, fontweight="bold")
    fig.tight_layout()
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        fig.savefig(output_path, dpi=150)
        print(f" Stages saved to: {output_path}")
    plt.show()
    plt.close(fig)


def plot_detections(image, bboxes, output_path=None):
    """Draws detected bounding boxes on the original image."""
    output = cv.cvtColor(image, cv.COLOR_GRAY2BGR)
    for x, y, w, h in bboxes:
        cv.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 2)
    output_rgb = cv.cvtColor(output, cv.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(output_rgb)
    ax.set_title(f"Detected Elements (OpenCV): {len(bboxes)} contours",
                 fontsize=14, fontweight="bold")
    ax.axis("off")
    fig.tight_layout()
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        fig.savefig(output_path, dpi=150)
        print(f"  Result saved to: {output_path}")
    plt.show()
    plt.close(fig)

## Run

In [ ]:
print("  GUI Element Detection — Classical Approach (OpenCV)")

image, stages, contours, bboxes = detect_gui_elements(
    IMAGE_PATH, CANNY_MIN, CANNY_MAX, BLUR_KERNEL, MORPH_KERNEL, MIN_AREA,
)

print(f"\n  Image: {IMAGE_PATH}")
print(f"  Dimensions: {image.shape[1]}x{image.shape[0]} pixels")
print(f"  Contours found: {len(contours)}")
print(f"  Bounding boxes: {len(bboxes)}")

plot_stages(stages, output_path=f"{OUTPUT_DIR}/opencv_stages.png")
plot_detections(image, bboxes, output_path=f"{OUTPUT_DIR}/opencv_detections.png")